В этом ноутбуке проводится сравнение нескольких моделей машинного обучения для задачи регрессии.

Целевая переменная: `payment_value`.

Основная метрика: RMSE.

Дополнительные метрики: MAE, R2.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore")

In [3]:
RANDOM_STATE = 42
DATA_PATH = "../data/processed/final_dataset.csv"
TARGET = "payment_value"

In [4]:
df = pd.read_csv(DATA_PATH)

df.head()

,customer_zip_code_prefix,order_item_id,price,freight_value,payment_sequential,payment_installments,payment_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix
0,3149,1.0,29.99,8.72,1.0,1.0,18.12,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0
1,3149,1.0,29.99,8.72,3.0,1.0,2.00,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0
2,3149,1.0,29.99,8.72,2.0,1.0,18.59,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0
3,47813,1.0,118.70,22.76,1.0,1.0,141.46,29.0,178.0,1.0,400.0,19.0,13.0,19.0,31570.0
4,75265,1.0,159.90,19.22,1.0,3.0,179.12,46.0,232.0,1.0,420.0,24.0,19.0,21.0,14840.0


In [5]:
print("Размер датасета:", df.shape)
print("Колонки:")
print(df.columns.tolist())

Размер датасета: (117250, 15)
Колонки:
['customer_zip_code_prefix', 'order_item_id', 'price', 'freight_value', 'payment_sequential', 'payment_installments', 'payment_value', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix']


In [6]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Для простоты оставляем только числовые признаки
X = X.select_dtypes(include=["number"])

print("Размер X:", X.shape)
print("Размер y:", y.shape)

Размер X: (117250, 14)
Размер y: (117250,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (93800, 14)
Test: (23450, 14)


In [8]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {
        "model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

In [9]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [10]:
results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(name, pipeline, X_test, y_test)
    results.append(metrics)
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

,model,RMSE,MAE,R2
4,Gradient Boosting,67.100274,28.848403,0.824069
3,Random Forest,67.848755,28.960986,0.820122
5,Extra Trees,69.092962,29.254759,0.813464
1,Ridge,88.329124,39.197344,0.695139
0,Linear Regression,88.329156,39.196975,0.695138
2,Lasso,88.389933,39.162555,0.694719


In [11]:
best_model_name = results_df.iloc[0]["model"]
best_rmse = results_df.iloc[0]["RMSE"]

print("Лучшая модель:", best_model_name)
print("RMSE:", best_rmse)

Лучшая модель: Gradient Boosting
RMSE: 67.10027354722516


In [12]:
ridge_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(random_state=RANDOM_STATE))
    ]
)

param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}

grid_search = GridSearchCV(
    ridge_pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print("Лучший CV RMSE:", -grid_search.best_score_)

Лучшие параметры: {'model__alpha': 10.0}
Лучший CV RMSE: 84.51950554986944


In [13]:
ridge_tuned_metrics = evaluate_model(
    "Ridge tuned",
    grid_search.best_estimator_,
    X_test,
    y_test
)

ridge_tuned_metrics

{'model': 'Ridge tuned',
 'RMSE': 88.32883182126369,
 'MAE': 39.200666067213874,
 'R2': 0.6951405780187403}

In [14]:
final_results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([ridge_tuned_metrics])
    ],
    ignore_index=True
).sort_values("RMSE")

final_results_df

,model,RMSE,MAE,R2
0,Gradient Boosting,67.100274,28.848403,0.824069
1,Random Forest,67.848755,28.960986,0.820122
2,Extra Trees,69.092962,29.254759,0.813464
6,Ridge tuned,88.328832,39.200666,0.695141
3,Ridge,88.329124,39.197344,0.695139
4,Linear Regression,88.329156,39.196975,0.695138
5,Lasso,88.389933,39.162555,0.694719


## Вывод

В рамках экспериментов были обучены модели Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting и Extra Trees.

Основной метрикой выбрана RMSE, так как задача является задачей регрессии, а RMSE сильнее штрафует крупные ошибки прогноза стоимости заказа.

Лучшая модель выбирается по минимальному значению RMSE на тестовой выборке.